In [1]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import mlflow
import optuna
import datetime

from sklearn.model_selection import (
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank-Customer-Churn-Prediction-Experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1757658532427, experiment_id='1', last_update_time=1757658532427, lifecycle_stage='active', name='Bank-Customer-Churn-Prediction-Experiment', tags={}>

# Data preprocessing

In [3]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.head()

    return df


def split_data(df):
    # Train/val/test stratified split of ratio 0.8/0.1/0.1
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_vtest, y_train, y_vtest = train_test_split(
        df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_vtest, y_vtest, test_size=0.5, random_state=seed, stratify=y_vtest
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [4]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]


def preprocess_data(X_train, X_val, X_test):
    preprocessor = ColumnTransformer(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
            ("scaler", StandardScaler(), num),
        ]
    )

    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    with mlflow.start_run():
        mlflow.sklearn.log_model(preprocessor, "ChurnDataPreprocessor")

    return X_train, X_val, X_test, preprocessor

In [5]:
records = clean_data(data_path)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(records)
X_train_tf, X_val_tf, X_test_tf, pp = preprocess_data(X_train, X_val, X_test)

2025/09/14 07:44:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/14 07:44:25 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/09/14 07:44:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/09/14 07:44:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run caring-cow-420 at: http://localhost:5000/#/experiments/1/runs/dc7cca57beb8480f8c2a12061ede943d
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Model Evaluation and Hyperparameters Tuning

In [6]:
sampler = optuna.samplers.TPESampler(seed=seed)

In [10]:
def xgb_objective(trial):
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(X_train_tf, label=y_train)
        valid = xgb.DMatrix(X_val_tf, label=y_val)

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.1, 1.0),
            "max_depth": trial.suggest_int("max_depth", 1, 12),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1e-6, 500.0, log=True
            ),
            "seed": seed,
        }

        model = xgb.train(
            params,
            train,
            evals=[(valid, "validation")],
            early_stopping_rounds=300,
            verbose_eval=False,
        )

        preds = model.predict(valid)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)

    return roc_auc


def train_best_xgb_model(X_train, y_train, best_params):
    train = xgb.DMatrix(X_train, label=y_train)

    model = xgb.train(best_params, train)

    return model


def plot_feature_importance(model, feat_names=None):
    """
    Plots feature importance for an XGBoost model.

    Args:
    - model: A trained XGBoost model

    Returns:
    - fig: The matplotlib figure object
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_type = "gain"
    if feat_names is not None:
        model.feature_names = list(feat_names)

    xgb.plot_importance(
        model,
        importance_type=importance_type,
        ax=ax,
        title=f"Feature Importance based on {importance_type}",
    )
    plt.tight_layout()
    plt.close(fig)

    return fig


def hyperparameter_tuning(X_train, y_train, feat_names=None):
    with mlflow.start_run(
        run_name=f"xgboost_hyperparameter_tuning_{datetime.datetime.now().date()}",
        nested=True,
    ):
        mlflow.set_tag("model", "xgboost")
        study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
        study_xgb.optimize(xgb_objective, n_trials=200)

        print("Number of finished trials:", len(study_xgb.trials))
        print("Best value:", study_xgb.best_value)

        mlflow.log_params(study_xgb.best_params)

        xgb_model = train_best_xgb_model(X_train, y_train, study_xgb.best_params)

        mlflow.xgboost.log_model(
            xgb_model=xgb_model,
            name="mlflow_model",
            input_example=X_train[:5],
            registered_model_name="XGBoostChurnModel",
        )

        importances = plot_feature_importance(
            xgb_model,
            feat_names=feat_names,
        )
        mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

In [11]:
hyperparameter_tuning(
    X_train=X_train_tf, y_train=y_train, feat_names=pp.get_feature_names_out()
)

[I 2025-09-14 07:46:07,916] A new study created in memory with name: no-name-7d044ef8-1f10-49ee-a074-c9fd0c0baaf3
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:07] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-14 07:46:08,035] Trial 0 finished with value: 0.7070893684106809 and parameters: {'n_estimators': 1935, 'learning_rate': 0.7969454818643928, 'reg_lambda': 0.11270245072599183, 'reg_alpha': 0.0038480732119896897, 'subsample': 0.24041677639819287, 'max_depth': 2, 'max_delta_step': 0, 'min_child_weight': 9, 'gamma': 0.000169465562039471, 'scale_pos_weight': 1.4437836359206417}. Best is trial 0 with value: 0.7070893684106809.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pi

🏃 View run traveling-worm-283 at: http://localhost:5000/#/experiments/1/runs/8a33f37e9b1b4cec82eeab5fb3b158c0
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run placid-newt-901 at: http://localhost:5000/#/experiments/1/runs/0776d3962483456cae000045a909c03d
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run casual-gull-371 at: http://localhost:5000/#/experiments/1/runs/1c8257e6f6814520ae4c430b62eb89c6
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run fun-quail-631 at: http://localhost:5000/#/experiments/1/runs/9f34720afb2b44a6b8fee4ff438010f4
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:08] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-14 07:46:08,260] Trial 4 finished with value: 0.5 and parameters: {'n_estimators': 698, 'learning_rate': 0.09780337016659407, 'reg_lambda': 2.3893167752763565e-09, 'reg_alpha': 10.058296249730986, 'subsample': 0.33290198344001526, 'max_depth': 8, 'max_delta_step': 3, 'min_child_weight': 6, 'gamma': 5.699231800614222e-05, 'scale_pos_weight': 4.0554902701197996e-05}. Best is trial 0 with value: 0.7070893684106809.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:08] WARNING: /Users/runner/minifor

🏃 View run languid-turtle-679 at: http://localhost:5000/#/experiments/1/runs/dcbda3af6c0e443b8fba184579b65688
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run luminous-goose-646 at: http://localhost:5000/#/experiments/1/runs/5a6728d22b814ef692079792118ea718
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run thundering-ant-505 at: http://localhost:5000/#/experiments/1/runs/80c4d7eb6eb44486bd6137175831ea58
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run defiant-bird-962 at: http://localhost:5000/#/experiments/1/runs/c8aa73cb14d749efa0d78e62354feb03
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:08] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-14 07:46:08,501] Trial 8 finished with value: 0.5 and parameters: {'n_estimators': 4330, 'learning_rate': 0.17643967683381545, 'reg_lambda': 4.363935001509045e-06, 'reg_alpha': 5.001978874034509e-09, 'subsample': 0.37988408954409597, 'max_depth': 4, 'max_delta_step': 8, 'min_child_weight': 7, 'gamma': 0.05222002015831429, 'scale_pos_weight': 0.012816913980027161}. Best is trial 0 with value: 0.7070893684106809.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:08] WARNING: /Users/runner/miniforg

...

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:22] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-14 07:46:22,352] Trial 198 finished with value: 0.750036949453148 and parameters: {'n_estimators': 1093, 'learning_rate': 0.5649452141073311, 'reg_lambda': 9.925603573074257, 'reg_alpha': 3.5633815064369796e-07, 'subsample': 0.34065495987200867, 'max_depth': 3, 'max_delta_step': 10, 'min_child_weight': 8, 'gamma': 0.07144509275241318, 'scale_pos_weight': 8.721696924563222}. Best is trial 163 with value: 0.7955709922159819.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [07:46:22] WARNING: /Users/run

🏃 View run skittish-boar-692 at: http://localhost:5000/#/experiments/1/runs/d6199fa5773843139517ac0bbb5188fd
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run handsome-croc-743 at: http://localhost:5000/#/experiments/1/runs/af97432fe3e8451abe0d86288b0c054c
🧪 View experiment at: http://localhost:5000/#/experiments/1
Number of finished trials: 200
Best value: 0.7955709922159819


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/mlflow/xgboost/__init__.py:168: UserWarning: [07:46:22] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/09/14 07:46:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'XGBoostChurnModel' already exists. Creating a new version of this model...
2025/09/14 07:46:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoostChurnModel, version 2


🏃 View run xgboost_hyperparameter_tuning_2025-09-14 at: http://localhost:5000/#/experiments/1/runs/4097fda16b3842859ecab729c0152282
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '2' of model 'XGBoostChurnModel'.


In [14]:
loaded_preprocessor = mlflow.sklearn.load_model(f"models:/ChurnDataPreprocessor/latest")
loaded_model = mlflow.xgboost.load_model(f"models:/XGBoostChurnModel/latest")

In [15]:
def make_predictions(
    model, X_test: pd.DataFrame, preprocessor: ColumnTransformer
) -> np.ndarray:
    X_test = preprocessor.transform(X_test)
    preds = model.predict(xgb.DMatrix(X_test))
    return np.clip(np.rint(preds), 0, 1).astype(int)

In [16]:
preds = make_predictions(loaded_model, X_test, loaded_preprocessor)

In [19]:
preds_tr = make_predictions(loaded_model, X_train, loaded_preprocessor)
X_train = X_train.assign(Preds=preds_tr)
preds_val = make_predictions(loaded_model, X_val, loaded_preprocessor)
X_val = X_val.assign(Preds=preds_val)
preds_t = make_predictions(loaded_model, X_test, loaded_preprocessor)
X_test = X_test.assign(Preds=preds_t)

In [20]:
X_trv = pd.concat([X_train, X_val], axis=0)
X_trv

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Complain,Satisfaction Score,Card Type,Point Earned,Preds
4791,4792,15746461,Taylor,709,Spain,Male,35,2,0.00,2,1,0,104982.39,0,2,GOLD,422,0
8881,8882,15618647,Kornilova,744,France,Male,29,1,43504.42,1,1,1,119327.75,0,1,PLATINUM,607,0
6166,6167,15567431,Kodilinyechukwu,773,France,Male,64,2,145578.28,1,0,1,186172.85,0,1,SILVER,630,1
4473,4474,15713532,Wang,646,Germany,Female,29,4,105957.44,1,1,0,15470.91,0,1,PLATINUM,345,1
854,855,15601589,Baresi,675,France,Female,57,8,0.00,2,0,1,95463.29,0,3,SILVER,632,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5392,5393,15710012,Bowen,738,Spain,Male,44,2,0.00,2,1,0,43018.82,1,5,PLATINUM,546,0
2328,2329,15664204,Meany,706,Spain,Male,29,2,0.00,2,1,1,18255.51,0,1,SILVER,868,0
6826,6827,15727361,Chiemela,547,France,Female,51,1,0.00,2,1,1,56908.41,0,2,SILVER,868,0
7511,7512,15686913,Kung,757,France,Male,38,0,0.00,1,1,0,83263.06,0,1,PLATINUM,537,1


# Evidently Report

In [34]:
from evidently import Report, DataDefinition, Dataset
from evidently.presets import DataDriftPreset
from evidently.ui.workspace import RemoteWorkspace
from evidently.metrics import ValueDrift, DriftedColumnsCount, MissingValueCount

ws = RemoteWorkspace("http://localhost:8000")

In [ ]:
if ws.search_project("Churn Prediction Project"):
    project = ws.get_project("Churn Prediction Project")
else:
    project = ws.create_project(name="Churn Prediction Project")

In [ ]:
data_definition = DataDefinition(
    numerical_columns=num,
    categorical_columns=cat + ["Exited", "Preds"],
)

cur_data = Dataset.from_pandas(
    data=X_train.assign(Exited=y_train),
    data_definition=data_definition,
)

ref_data = Dataset.from_pandas(
    data=X_test.assign(Exited=y_test),
    data_definition=data_definition,
)

report = Report(
    [
        ValueDrift(column="Preds"),
        DriftedColumnsCount(),
        MissingValueCount(column="Preds"),
        DataDriftPreset(),
    ],
    include_tests=True,
)

eval = report.run(cur_data, reference_data=ref_data)

In [39]:
ws.add_run(project.id, eval)

Report ID: 019945da-f8c1-79cf-ae26-84881fabc702
Link: http://localhost:8000/projects/019945ce-0ed4-73c5-befe-0e0b207ce1d9/reports/019945da-f8c1-79cf-ae26-84881fabc702

In [40]:
eval.dict()["metrics"]

[{'id': 'f989471f535449d947dc1594bee11bbe',
  'metric_id': 'ValueDrift(column=Preds)',
  'value': np.float64(0.12755775736676545)},
 {'id': '15e89f895b482f9b84ba7274ed18a106',
  'metric_id': 'DriftedColumnsCount(drift_share=0.5)',
  'value': {'count': 3.0, 'share': 0.2}},
 {'id': '4c3d37d67814b7823c164fbf505544bc',
  'metric_id': 'MissingValueCount(column=Preds)',
  'value': {'count': 0.0, 'share': np.float64(0.0)}},
 {'id': 'c4f1631539c3955b53e20cab2fc9e838',
  'metric_id': 'ValueDrift(column=CreditScore)',
  'value': np.float64(0.9663582551858991)},
 {'id': '8f5d1c60a32d6fc1bd54bc53af61d8e8',
  'metric_id': 'ValueDrift(column=Age)',
  'value': np.float64(0.4592764651990313)},
 {'id': '44af7ff50b9319ad30493a39cabd4056',
  'metric_id': 'ValueDrift(column=Tenure)',
  'value': np.float64(0.9830135526838559)},
 {'id': 'f11fe99e16eef1d959d624000c604160',
  'metric_id': 'ValueDrift(column=Balance)',
  'value': np.float64(0.4936461401575463)},
 {'id': '40bb5b6d43598f6e48fe609e4d9cb46c',
  'm

In [42]:
results = eval.dict()
prediction_drift = results["metrics"][0]["value"]
num_drifted_columns = results["metrics"][1]["value"]["count"]
share_missing_values = results["metrics"][2]["value"]["share"]